# Wairagkar 2025 · voice synthesis T15

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `t15_neural_data.zip` from [Dryad](https://doi.org/10.5061/dryad.2280gb64f). Choose the May 8, 2026 release. Create `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/` and save the download as `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip`.

### 2. Extract for inspection (optional)

Extract the archive into `studies/20250612_wairagkar/data/extracted/dryad_2280gb64f/`.

```text
python -m zipfile -e "studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip" "studies/20250612_wairagkar/data/extracted/dryad_2280gb64f"
```

After extraction, check for `studies/20250612_wairagkar/data/extracted/dryad_2280gb64f/Dryad_T15_all/t15_day025/`.

### 3. Prepare the model input

The inference and training scripts read neural data and acoustic targets directly from `studies/20250612_wairagkar/data/raw/dryad_2280gb64f/dryad_2280gb64f__t15_neural_data.zip`. They prepare arrays in memory. The ZIP is the pipeline's input.

Set up the upstream encoder and vocoder code before running the pipeline.

```text
python shared/tools/setup_upstream.py --study 20250612_wairagkar
```

## Read the archived results

The acoustic output is continuous speech. The reproduction archives per-day mean Mel-band correlations. The paper's listener word-transcription median is a separate Figure 2c value.

This walkthrough uses the checked-in archive and takes no data download.

In [1]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [2]:
voice = root / "studies/20250612_wairagkar/results/20261124_import_voice_synthesis/tier2_mel_full_trial.json"
results = json.loads(voice.read_text(encoding="utf-8"))
for day, metrics in results["days"].items():
    r = metrics["mel_band_r_mean"]
    print(f"{day}: {metrics['n_scored']} scored trials, r = {r:.3f}, 100 × (1 − r) = {100 * (1-r):.1f}%")
paper = next(row for row in benchmark["panel_cd_online_wer_speed"] if row["label"].startswith("Wairagkar"))
print(f"Paper's separate listener median: {paper['wer_mean']}% WER")

t15_day179: 20 scored trials, r = 0.612, 100 × (1 − r) = 38.8%
t15_day195: 20 scored trials, r = 0.686, 100 × (1 − r) = 31.4%
Paper's separate listener median: 43.75% WER


## Continue the full reproduction

`t15_control_experiment.zip` is a separate release file.

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`infer.py`](code/infer.py), [`evaluate.py`](code/evaluate.py). Run a command with `--help` to see its arguments.